# 9장. 토큰화

| 항목 | 내용 |
|---|---|
| 예상 시간 | 5시간 |
| 선수 장 | 8장 |
| 이 장의 결과물 | 영어·한국어 토큰화 함수, Okt vs 공백 토큰화 비교표, 리포트용 "성능이 왜 달라지는가" |
| 관련 학습 목표 | ① 토큰화의 목적과 언어별 처리 방식의 차이 |

**학습 목표**
- 토큰화가 왜 필요하고, 토큰 단위가 결과에 어떤 영향을 주는지 설명할 수 있다.
- 영어를 공백·정규식·NLTK로 토큰화하고, 어간 추출의 장단점을 말할 수 있다.
- 한국어가 교착어라서 공백 토큰화가 약한 이유를 데이터로 보여 줄 수 있다.
- KoNLPy Okt로 형태소 분석을 하고 품사로 거를 수 있다.
- 공백 토큰화와 형태소 분석을 어휘 크기, OOV 비율, 분류 성능으로 비교할 수 있다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

import time
from collections import Counter

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")


def clean_en(text: str) -> str:   # 8장
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:   # 8장
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


en = load_english()
ko = load_korean()
print("영어:", en.shape, "| 한국어:", ko.shape)

영어: (4797, 2) | 한국어: (7000, 3)


## 9.1 토큰화의 목적

**토큰화(tokenization)**는 문장을 **셀 수 있는 단위(토큰)**로 자르는 일입니다. BoW와 TF-IDF는 토큰을 세는 방법이므로, *무엇을 한 단위로 볼지*가 모든 결과를 좌우합니다.

좋은 토큰 단위의 조건:
1. **같은 의미는 같은 토큰**이 되어야 합니다. `학교에`, `학교를`, `학교가`가 모두 `학교`로 세어져야 "학교"에 관한 문서끼리 비슷해집니다.
2. **다른 의미는 다른 토큰**이어야 합니다. 너무 잘게 자르면(예: 글자 단위) 의미가 사라집니다.

언어마다 단어가 만들어지는 방식이 달라서, 좋은 토큰화 방법도 다릅니다.

## 9.2 영어 토큰화

영어는 단어가 **공백으로 이미 분리**되어 있어서 비교적 쉽습니다. 세 가지 방법을 비교합니다.

In [2]:
from nltk.tokenize import word_tokenize

sentences = [
    "NASA's new rocket didn't launch; it costs $1.5B.",
    "The Yankees won 5-3, and the fans weren't happy!",
]
for s in sentences:
    print(s)
    print("  split()       :", s.split())
    print("  정제 + split  :", clean_en(s).split())
    print("  word_tokenize :", word_tokenize(s))

NASA's new rocket didn't launch; it costs $1.5B.
  split()       : ["NASA's", 'new', 'rocket', "didn't", 'launch;', 'it', 'costs', '$1.5B.']
  정제 + split  : ["nasa's", 'new', 'rocket', "didn't", 'launch', 'it', 'costs', 'b']
  word_tokenize : ['NASA', "'s", 'new', 'rocket', 'did', "n't", 'launch', ';', 'it', 'costs', '$', '1.5B', '.']
The Yankees won 5-3, and the fans weren't happy!
  split()       : ['The', 'Yankees', 'won', '5-3,', 'and', 'the', 'fans', "weren't", 'happy!']
  정제 + split  : ['the', 'yankees', 'won', 'and', 'the', 'fans', "weren't", 'happy']
  word_tokenize : ['The', 'Yankees', 'won', '5-3', ',', 'and', 'the', 'fans', 'were', "n't", 'happy', '!']


| 방법 | 장점 | 단점 |
|---|---|---|
| `split()` | 가장 빠르고 단순 | 문장부호가 단어에 붙는다 (`launch;`) |
| 정제 + `split()` (8장 정규식) | 빠르고, 정제 규칙과 한 번에 처리 | 규칙을 직접 설계해야 한다 |
| NLTK `word_tokenize` | 영어 문법 규칙(`did`+`n't`)을 안다 | 느리고, 문장부호·숫자도 토큰으로 남아 따로 걸러야 한다 |

속도도 비교해 봅니다.

In [3]:
sample_texts = en["text"].head(500).tolist()

t0 = time.perf_counter()
a = [clean_en(t).split() for t in sample_texts]
t1 = time.perf_counter()
b = [word_tokenize(t.lower()) for t in sample_texts]
t2 = time.perf_counter()
print(f"정제 + split  : {t1 - t0:.2f}초")
print(f"word_tokenize : {t2 - t1:.2f}초 ({(t2 - t1) / (t1 - t0):.0f}배)")

정제 + split  : 0.01초
word_tokenize : 0.23초 (15배)


주제 분류와 검색에서는 `didn't`를 `did`+`n't`로 나누는 정교함이 큰 차이를 만들지 않습니다. 그래서 영어는 **8장 정제 + 공백 기반 토큰화**로 충분합니다. 과제 문서도 영어는 "NLTK 또는 공백 기반"을 허용합니다.

## 9.3 영어 정규화: 어간 추출과 표제어 추출

`computer`, `computers`, `computing`은 같은 개념인데 다른 토큰입니다. 이를 합치는 방법이 두 가지 있습니다.

| 방법 | 원리 | 예 |
|---|---|---|
| **어간 추출(stemming)** | 규칙으로 어미를 잘라낸다. 빠르지만 사전에 없는 형태가 나온다 | `studies` → `studi`, `university` → `univers` |
| **표제어 추출(lemmatization)** | 사전과 품사로 원형을 찾는다. 정확하지만 느리고 품사 정보가 필요하다 | `studies` → `study`, `better` → `good` |

In [4]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
words = ["computer", "computers", "computing", "computation", "studies", "studying",
         "university", "universe", "graphics", "graphic", "launched", "launches"]
for w in words:
    print(f"{w:<12} → {stemmer.stem(w)}")

computer     → comput
computers    → comput
computing    → comput
computation  → comput
studies      → studi
studying     → studi
university   → univers
universe     → univers
graphics     → graphic
graphic      → graphic
launched     → launch
launches     → launch


`university`와 `universe`가 같은 어간 `univers`가 되는 것처럼 **다른 단어가 합쳐지는** 부작용도 있습니다. 실제 데이터에서 어휘가 얼마나 줄어드는지 확인합니다.

In [5]:
en_tokens = [clean_en(t).split() for t in en["text"]]
vocab = Counter(t for toks in en_tokens for t in toks)
stem_of = {w: stemmer.stem(w) for w in vocab}

print(f"어휘 수: {len(vocab):,} → 어간 추출 후 {len(set(stem_of.values())):,}")

groups = {}
for w, s in stem_of.items():
    groups.setdefault(s, []).append(w)
for s in ["comput", "launch", "orbit"]:
    print(f"'{s}' 로 합쳐진 단어:", sorted(groups.get(s, []), key=lambda w: -vocab[w])[:8])

어휘 수: 36,444 → 어간 추출 후 27,473
'comput' 로 합쳐진 단어: ['computer', 'computing', 'computers', 'compute', 'computational', 'computation', 'computed', 'computes']
'launch' 로 합쳐진 단어: ['launch', 'launched', 'launches', 'launching', 'launchings']
'orbit' 로 합쳐진 단어: ['orbit', 'orbital', 'orbiter', 'orbiting', 'orbits', 'orbiters', 'orbited', 'orbitals']


**이 과제의 선택: 어간 추출을 하지 않는다.** 이유는
- 검색 결과와 분류 근거(20장에서 "어떤 단어 때문에 이렇게 예측했나")를 **사람이 읽을 수 있는 단어**로 보여 주기 위해서입니다. `univers`, `comput`은 설명하기 어렵습니다.
- 대가로 `work`와 `works`가 다른 단어가 됩니다. 이 손실은 21장 BoW 한계 분석에서 사례로 다룹니다.

어간 추출을 쓰기로 해도 괜찮습니다. 다만 그 경우 근거와 함께 분류 성능 변화를 측정해서 리포트에 쓰세요.

## 9.4 한국어의 특성: 왜 공백으로 자르면 안 될까

한국어는 **교착어**입니다. 의미를 가진 말(체언, 용언의 어간) 뒤에 **조사**와 **어미**가 붙어 한 덩어리(어절)를 만들고, 공백은 이 어절 사이에 들어갑니다.

```
삼성전자가    실적을    발표했다
[삼성전자+가] [실적+을] [발표+하+었+다]
  명사 조사    명사 조사  명사 동사 어미
```

공백으로 자르면 **같은 명사가 조사에 따라 다른 토큰**이 됩니다. 실제 데이터에서 확인해 봅니다.

In [6]:
ko_ws_tokens = [clean_ko(t).split() for t in ko["text"]]
ws_vocab = Counter(t for toks in ko_ws_tokens for t in toks)

JOSA = ["이", "가", "은", "는", "을", "를", "의", "에", "에서", "와", "과", "도", "로", "으로", "만", "까지", "에게", "서"]
for noun in ["미국", "한국", "스마트폰", "트럼프"]:
    forms = {noun + j: ws_vocab[noun + j] for j in JOSA if ws_vocab[noun + j] > 0}
    n_forms = sum(forms.values())
    print(f"{noun}: 조사 없이 {ws_vocab[noun]}회 / 조사가 붙은 변형 {len(forms)}종 {n_forms}회 "
          f"({n_forms / (ws_vocab[noun] + n_forms):.0%})")
    print("    ", forms)

미국: 조사 없이 44회 / 조사가 붙은 변형 9종 19회 (30%)
     {'미국이': 3, '미국은': 2, '미국을': 1, '미국의': 4, '미국에': 1, '미국과': 3, '미국도': 1, '미국으로': 1, '미국서': 3}
한국: 조사 없이 87회 / 조사가 붙은 변형 8종 17회 (16%)
     {'한국이': 1, '한국은': 5, '한국을': 1, '한국의': 3, '한국에': 2, '한국에서': 1, '한국도': 1, '한국서': 3}
스마트폰: 조사 없이 49회 / 조사가 붙은 변형 7종 11회 (18%)
     {'스마트폰이': 1, '스마트폰에': 3, '스마트폰과': 1, '스마트폰도': 1, '스마트폰으로': 3, '스마트폰만': 1, '스마트폰서': 1}
트럼프: 조사 없이 82회 / 조사가 붙은 변형 6종 15회 (15%)
     {'트럼프가': 3, '트럼프는': 1, '트럼프의': 2, '트럼프에': 3, '트럼프와': 5, '트럼프로': 1}


뉴스 제목은 조사를 생략하는 경우가 많은데도, `미국` 하나가 `미국이`, `미국의`, `미국과`… 최대 10개 열로 흩어지고 전체 등장의 상당 부분이 변형 쪽에 있습니다. 일반 문장(본문, 리뷰)이라면 조사가 훨씬 많이 붙어 문제가 더 커집니다. 그 결과
- 같은 개념의 문서 빈도가 여러 열로 **쪼개지고**,
- 검색어 `정부`가 `정부는`만 가진 문서를 **찾지 못하고**,
- 학습 때 `정부가`만 봤다면 테스트의 `정부에`는 **처음 보는 단어(OOV)**가 됩니다.

그래서 한국어는 **형태소 분석**으로 조사·어미를 떼어 내는 것이 좋습니다.

## 9.5 형태소 분석: KoNLPy Okt

**형태소**는 의미를 가진 가장 작은 단위입니다. 형태소 분석기는 어절을 형태소로 나누고 **품사**를 붙입니다. KoNLPy의 여러 분석기 중 **Okt**(Open Korean Text)는 설치가 쉽고 신조어·구어에 비교적 강해서 많이 씁니다(1장에서 Java 설치 확인).

| 메서드 | 결과 |
|---|---|
| `okt.morphs(text)` | 형태소 리스트 |
| `okt.nouns(text)` | 명사만 |
| `okt.pos(text)` | `(형태소, 품사)` 튜플 리스트 |
| 옵션 `norm=True` | 구어 표현 정규화 (`그랰ㅋㅋ` → `그래ㅋㅋ`) |
| 옵션 `stem=True` | 동사·형용사를 **원형**으로 (`발표했다` → `발표하다`) |

In [7]:
from konlpy.tag import Okt

okt = Okt()
s = "삼성전자가 3분기 사상 최대 실적을 발표했다"
print("morphs          :", okt.morphs(s))
print("nouns           :", okt.nouns(s))
print("pos             :", okt.pos(s))
print("pos(stem=True)  :", okt.pos(s, norm=True, stem=True))

morphs          : ['삼', '성', '전자', '가', '3분', '기', '사상', '최대', '실적', '을', '발표', '했다']
nouns           : ['전자', '사상', '최대', '실적', '발표']
pos             : [('삼', 'Modifier'), ('성', 'Modifier'), ('전자', 'Noun'), ('가', 'Josa'), ('3분', 'Number'), ('기', 'Foreign'), ('사상', 'Noun'), ('최대', 'Noun'), ('실적', 'Noun'), ('을', 'Josa'), ('발표', 'Noun'), ('했다', 'Verb')]
pos(stem=True)  : [('삼', 'Modifier'), ('성', 'Modifier'), ('전자', 'Noun'), ('가', 'Josa'), ('3분', 'Number'), ('기', 'Foreign'), ('사상', 'Noun'), ('최대', 'Noun'), ('실적', 'Noun'), ('을', 'Josa'), ('발표', 'Noun'), ('하다', 'Verb')]


`stem=True`를 주면 `했다`가 원형 `하다`가 됩니다. 그런데 `삼성전자가`는 `삼`, `성`(Modifier, 수식어)과 `전자`로 잘못 쪼개졌습니다. 형태소 분석기도 틀릴 수 있다는 점은 9.6절에서 다시 봅니다. "발표했다", "발표한", "발표하고"가 모두 `하다`로 모입니다.

### 실제 제목에서 나오는 품사
7,000개 제목 전체를 분석해서 **어떤 품사가 얼마나 나오는지** 셉니다. 한 번 분석한 결과는 이후 셀에서 다시 씁니다(약 10~20초 걸립니다).

In [8]:
t0 = time.perf_counter()
ko_pos = [okt.pos(clean_ko(t), norm=True, stem=True) for t in ko["text"]]
print(f"Okt 분석 시간: {time.perf_counter() - t0:.1f}초 ({len(ko_pos):,}건)")

pos_counts = Counter(p for doc in ko_pos for _, p in doc)
examples = {}
for doc in ko_pos:
    for w, p in doc:
        examples.setdefault(p, Counter())[w] += 1

rows = [(p, c, ", ".join(w for w, _ in examples[p].most_common(6))) for p, c in pos_counts.most_common()]
pd.DataFrame(rows, columns=["품사", "개수", "자주 나오는 예"])

Okt 분석 시간: 6.3초 (7,000건)


,품사,개수,자주 나오는 예
0,Noun,50482,"종합, 대통령, 한국, 삼성, 위, 명"
1,Josa,5066,"에, 의, 서, 로, 은, 으로"
2,Number,4629,"2, 5, 1, 3, 4, 8"
3,Verb,3632,"하다, 되다, 대다, 보다, 개다, 않다"
4,Alpha,2388,"G, LG, KT, SK, S, AI"
5,Foreign,1576,"美, 北, 中, 기, 朴, 靑"
6,Modifier,1321,"기, 신, 대, 전, 동, 문"
7,Suffix,1206,"들, 적, 장, 화, 당, 형"
8,Adjective,944,"없다, 있다, 밉다, 많다, 좋다, 아니다"
9,Adverb,240,"우, 없이, 함께, 다, 씽, 풍"


## 9.6 품사 필터 설계

모든 형태소를 쓰면 조사(`Josa`), 어미(`Eomi`), 구두점이 단어가 됩니다. 이것들은 모든 주제에 고르게 나와서 주제를 가르지 못합니다. 위 표를 보고 **남길 품사**를 정합니다.

| 품사 | 판단 | 근거 |
|---|---|---|
| `Noun` 명사 | ✅ 남김 | 주제를 나타내는 핵심 (정부, 실적, 경기) |
| `Verb` 동사, `Adjective` 형용사 | ✅ 남김 | `stem=True`로 원형이 되어 합쳐짐. "출시하다", "급등하다"도 주제 단서 |
| `Alpha` 영문 | ✅ 남김 | `AI`, `SKT`, `LG` 같은 기업·기술명 |
| `Foreign` 중 한자 | ✅ 남김 | 8장에서 본 `北`, `美` 등 |
| `Josa` 조사, `Eomi` 어미, `Punctuation` | ❌ 버림 | 기능어, 주제 신호 없음 |
| `Number` 숫자 | ❌ 버림 | 값이 제각각 (8장과 같은 이유) |
| `Suffix` 접미사, `Modifier` 수식어, `Determiner` 관형사 등 | ❌ 버림 | 단독으로는 의미가 약함 |

> **명사만 쓰면 안 될까?** 명사만 써도 꽤 잘 됩니다. 하지만 "급등하다", "출시하다"처럼 경제·IT 주제를 가르는 동사가 빠지고, 영문·한자 약어를 따로 챙겨야 합니다. 여기서는 동사·형용사까지 남기는 쪽을 택하고, 쓸모없는 용언(`하다`, `되다`, `있다`)은 10장 불용어로 지웁니다.

In [9]:
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}


def filter_pos(pos_list: list[tuple[str, str]]) -> list[str]:
    return [w.lower() for w, p in pos_list
            if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]


for i in range(4):
    print(ko.loc[i, "text"])
    print("  공백  :", ko_ws_tokens[i])
    print("  Okt   :", filter_pos(ko_pos[i]))

朴대통령 지방·중앙이 힘모아 위기를 기회로
  공백  : ['朴대통령', '지방', '중앙이', '힘모아', '위기를', '기회로']
  Okt   : ['朴', '대통령', '지방', '중앙', '힘', '모으다', '위기', '기회']
셀카·합성영상은 성폭력처벌법 적용 안 돼…보완책 시급
  공백  : ['셀카', '합성영상은', '성폭력처벌법', '적용', '안', '돼', '보완책', '시급']
  Okt   : ['셀카', '합성', '영상', '성폭력', '처벌', '법', '적용', '안', '돼다', '보완', '책', '시급']
아모레퍼시픽 소비자단체와 이해관계자 포럼 열어
  공백  : ['아모레퍼시픽', '소비자단체와', '이해관계자', '포럼', '열어']
  Okt   : ['아모레퍼시픽', '소비자', '단체', '이해관계자', '포럼', '열다']
KT 데이터 공유 앱 Y데이터박스 가입자 10만 돌파
  공백  : ['KT', '데이터', '공유', '앱', 'Y데이터박스', '가입자', '10만', '돌파']
  Okt   : ['kt', '데이터', '공유', '앱', 'y', '데이터', '박스', '가입자', '돌파']


### 형태소 분석기도 틀린다
Okt는 사전과 통계 규칙으로 동작하므로 **고유명사나 신조어를 엉뚱하게 자르기도** 합니다.

In [10]:
for s in ["삼성전자 갤럭시노트7 단종", "SKT버라이즌 5G 공동연구", "손흥민 시즌 첫 골 쏜다"]:
    print(f"{s:<20} → {filter_pos(okt.pos(clean_ko(s), norm=True, stem=True))}")

삼성전자 갤럭시노트7 단종       → ['삼성', '전자', '갤럭시', '노트', '단종']
SKT버라이즌 5G 공동연구      → ['skt', '버라이즌', 'g', '공동', '연구']
손흥민 시즌 첫 골 쏜다        → ['손흥민', '시즌', '첫', '골', '쏜다']


`삼성전자`가 문맥에 따라 `삼성`+`전자`로 나뉘기도 하고, 9.5절처럼 `삼`, `성`, `전자`로 잘려 `삼성`이 아예 사라지기도 합니다. `5G`는 `g`만 남습니다. 완벽하지 않지만, 조사 때문에 같은 명사가 수십 개로 쪼개지는 공백 토큰화보다는 훨씬 낫습니다. 이런 오류는 20장 오분류 분석에서 원인으로 등장할 수 있으니 기억해 두세요.

## 9.7 공백 vs Okt 비교 실험 ⭐

과제 조건: *"형태소 분석기 설치가 어려운 경우 공백 토큰화를 쓰되 성능이 왜 달라지는지 분석한다."* 두 방법을 **같은 데이터, 같은 분할, 같은 모델**로 비교해서 이 질문에 숫자로 답합니다.

| 지표 | 의미 |
|---|---|
| 어휘 수 | 서로 다른 토큰 수. 클수록 같은 개념이 쪼개졌을 가능성 |
| 한 번만 나온 토큰 비율 | 클수록 학습에 쓸 수 없는 희귀 토큰이 많음 |
| Test OOV 비율 | Test 토큰 중 Train 어휘에 없는 비율. 모델이 **아무 정보도 얻지 못하는** 토큰 |
| 분류 정확도·F1 | 미리 보기 (17–18장에서 직접 만든다) |

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

ko_okt_tokens = [filter_pos(p) for p in ko_pos]
ko_ws_lower = [[t.lower() for t in toks] for toks in ko_ws_tokens]
labels = ko["label"].to_numpy()
idx_tr, idx_te = train_test_split(np.arange(len(ko)), test_size=0.2, stratify=labels,
                                  random_state=RANDOM_STATE)


def compare(tokens: list[list[str]]) -> dict:
    counts = Counter(t for toks in tokens for t in toks)
    train_vocab = {t for i in idx_tr for t in tokens[i]}
    test_tokens = [t for i in idx_te for t in tokens[i]]
    vec = TfidfVectorizer(analyzer=lambda toks: toks, min_df=2)     # 미리 보기: 이미 자른 토큰을 그대로 사용
    X_tr = vec.fit_transform([tokens[i] for i in idx_tr])
    X_te = vec.transform([tokens[i] for i in idx_te])
    pred = LogisticRegression(max_iter=1000).fit(X_tr, labels[idx_tr]).predict(X_te)
    return {
        "어휘 수": len(counts),
        "한 번만 나온 토큰 비율": f"{np.mean([c == 1 for c in counts.values()]):.1%}",
        "문서당 토큰 수": round(np.mean([len(t) for t in tokens]), 1),
        "Test OOV 비율": f"{np.mean([t not in train_vocab for t in test_tokens]):.1%}",
        "정확도 (미리 보기)": round(accuracy_score(labels[idx_te], pred), 3),
        "macro F1 (미리 보기)": round(f1_score(labels[idx_te], pred, average="macro"), 3),
    }


comparison = pd.DataFrame({"공백 토큰화": compare(ko_ws_lower), "Okt 형태소 + 품사 필터": compare(ko_okt_tokens)})
comparison

,공백 토큰화,Okt 형태소 + 품사 필터
어휘 수,22725,11403
한 번만 나온 토큰 비율,71.6%,44.0%
문서당 토큰 수,7.2,8.4
Test OOV 비율,34.8%,10.5%
정확도 (미리 보기),0.708,0.767
macro F1 (미리 보기),0.712,0.767


**결과 해석 (리포트에 쓸 내용)**
- 공백 토큰화는 조사·어미가 붙은 어절을 그대로 쓰므로 **어휘가 훨씬 크고**, 대부분이 한 번만 나오는 희귀 토큰입니다.
- 그 결과 **Test 토큰 중 Train에서 본 적 없는 비율(OOV)이 크게** 늘어납니다. 처음 보는 토큰은 TF-IDF 벡터에서 0이 되므로 분류에 아무 기여도 못합니다.
- 짧은 제목에서 주제어 하나가 OOV가 되면 그 문서는 단서를 거의 잃습니다. 그래서 **분류 성능이 떨어집니다.**
- 형태소 분석은 `정부가/정부는/정부의`를 `정부`로 모아 문서 빈도를 한 열에 집중시킵니다.

> 이 표는 불용어 제거 전이고 sklearn을 쓴 미리 보기입니다. 10장(불용어), 13장(직접 구현한 TF-IDF), 18장(분류)을 거치면 숫자는 조금 달라지지만 **차이의 방향과 이유**는 같습니다.

## 9.8 형태소 분석 결과 캐시

Okt 분석은 7,000건에 10초 이상 걸립니다. 검색할 때마다, 실험할 때마다 다시 분석하면 시간이 낭비되므로 결과를 **파일로 저장(캐시)**해 두고 다시 씁니다.

주의할 점: 품사 목록이나 정제 규칙을 바꿨는데 **예전 캐시를 그대로 쓰면** 바뀐 규칙이 반영되지 않습니다. 그래서 캐시 파일 이름에 **설정의 지문(해시)**을 넣어, 설정이 바뀌면 다른 파일을 쓰게 합니다.

In [12]:
import hashlib
import json
import tempfile

CACHE_DIR = Path(tempfile.mkdtemp())    # 실습용. 실제 프로젝트에서는 data/cache/ (git 제외)


def cache_signature(keep_pos: set[str], okt_options: dict) -> str:
    config = {"keep_pos": sorted(keep_pos), "okt": okt_options, "clean": KO_CLEAN.pattern}
    return hashlib.sha1(json.dumps(config, sort_keys=True).encode()).hexdigest()[:10]


def okt_tokens_cached(texts: list[str], keep_pos=KO_KEEP_POS, okt_options={"norm": True, "stem": True}):
    path = CACHE_DIR / f"okt_{cache_signature(keep_pos, okt_options)}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), path.name, "캐시 사용"
    tokens = [filter_pos(okt.pos(clean_ko(t), **okt_options)) for t in texts]
    path.write_text(json.dumps(tokens, ensure_ascii=False), encoding="utf-8")
    return tokens, path.name, "새로 분석"


texts = ko["text"].tolist()
for _ in range(2):
    t0 = time.perf_counter()
    toks, name, how = okt_tokens_cached(texts)
    print(f"{how}: {time.perf_counter() - t0:.2f}초 ({name})")

_, name, how = okt_tokens_cached(texts[:10], keep_pos={"Noun"})
print(f"품사 설정을 바꾸면 → {how} ({name})")

새로 분석: 5.76초 (okt_f0ae29d338.json)
캐시 사용: 0.01초 (okt_f0ae29d338.json)
품사 설정을 바꾸면 → 새로 분석 (okt_76af9ea582.json)


> 이 예제는 텍스트 목록이 같다고 가정합니다. 실제로는 문서 ID(`guid`)를 키로 저장하거나, 데이터가 바뀌면 캐시를 지우는 규칙이 필요합니다.

## 📝 리포트용 정리: 언어별 토큰화 (학습 목표 ①)

| | 영어 | 한국어 |
|---|---|---|
| 언어 특성 | 굴절어. 단어가 공백으로 분리되고 형태 변화가 적다 (`work/works`) | 교착어. 명사·어간에 조사·어미가 붙어 어절을 이룬다 |
| 선택한 방법 | 정제(정규식) + 공백 토큰화, 어간 추출 안 함 | Okt 형태소 분석 (`norm`, `stem`) + 품사 필터 |
| 이유 | 공백만으로도 단어 단위가 거의 맞다. 어간은 사람이 읽기 어렵다 | 공백으로 자르면 같은 명사가 조사별로 쪼개져 어휘 폭증, OOV 증가 |
| 공백 토큰화를 쓰면 | 큰 문제 없음 | 어휘·OOV가 크게 늘고 분류 성능 하락 (9.7 표) |
| 알려진 한계 | `work/works` 분리 | 고유명사·신조어 오분석 (`삼성전자` → `삼성`+`전자`) |

## ✍️ 실습: 토큰화 함수 만들기

`tokenize_ko(text, method)`를 작성하세요.
- `method="whitespace"`: 8장 정제 후 공백으로 자르고 영문은 소문자로
- `method="okt"`: 8장 정제 후 Okt(`norm=True, stem=True`)로 분석하고 9.6의 품사 필터 적용
- 그 밖의 `method`는 `ValueError`

그리고 `'北, 미사일 발사…정부는 "강력 대응하겠다"'`를 두 방법으로 토큰화해 결과를 비교하세요.

In [13]:
# 여기에 작성하세요

**정답**

In [14]:
def tokenize_ko(text: str, method: str = "okt") -> list[str]:
    cleaned = clean_ko(text)
    if method == "whitespace":
        return [t.lower() for t in cleaned.split()]
    if method == "okt":
        return filter_pos(okt.pos(cleaned, norm=True, stem=True))
    raise ValueError(f"method는 'okt' 또는 'whitespace': {method}")


s = '北, 미사일 발사…정부는 "강력 대응하겠다"'
print("whitespace:", tokenize_ko(s, "whitespace"))
print("okt       :", tokenize_ko(s, "okt"))
try:
    tokenize_ko(s, "jieba")
except ValueError as e:
    print("ValueError:", e)

whitespace: ['北', '미사일', '발사', '정부는', '강력', '대응하겠다']
okt       : ['北', '미사일', '발사', '정부', '강력', '대응', '하다']
ValueError: method는 'okt' 또는 'whitespace': jieba


## 🧐 퀴즈

**Q1.** 한국어에서 공백 토큰화를 쓰면 Test OOV 비율이 높아지는 이유는?
<details><summary>▶ 정답 보기</summary>

한국어는 명사 뒤에 조사가 붙어 어절이 되므로, 같은 명사가 `정부가`, `정부는`, `정부의`처럼 여러 토큰으로 나뉩니다. Train에서 `정부가`만 봤다면 Test의 `정부에`는 처음 보는 토큰(OOV)이 되어 벡터에서 0이 됩니다.
</details>

**Q2.** Okt의 `stem=True` 옵션은 무엇을 하고, 왜 유용한가요?
<details><summary>▶ 정답 보기</summary>

동사·형용사를 원형으로 바꿉니다(`발표했다` → `발표하다`). 활용형이 달라도 같은 토큰으로 모여서 문서 빈도가 한 열에 집중됩니다.
</details>

**Q3.** 영어에서 어간 추출을 하지 않기로 했을 때의 장점과 대가는?
<details><summary>▶ 정답 보기</summary>

장점: 검색 결과와 분류 근거 단어를 사람이 읽을 수 있는 형태로 보여 줄 수 있고, `university`/`universe` 같은 잘못된 합침이 없습니다. 대가: `work`와 `works`처럼 같은 개념의 변형이 다른 토큰으로 남습니다.
</details>

**Q4.** 형태소 분석 결과를 캐시할 때, 캐시 파일 이름에 설정의 해시를 넣는 이유는?
<details><summary>▶ 정답 보기</summary>

품사 목록이나 정제 규칙을 바꿨는데 예전 캐시를 그대로 읽으면 바뀐 설정이 반영되지 않습니다. 설정이 바뀌면 해시가 달라져 다른 파일 이름이 되므로 자동으로 다시 분석합니다.
</details>

**Q5.** 조사(`Josa`)와 어미(`Eomi`)를 품사 필터에서 버리는 이유는?
<details><summary>▶ 정답 보기</summary>

모든 주제의 문장에 고르게 나오는 기능어라 주제를 가르는 정보가 없고, 모든 문서 벡터에 조금씩 들어가 유사도를 흐리기 때문입니다.
</details>

## ✅ 체크포인트
- [ ] 토큰 단위가 검색·분류 결과에 미치는 영향을 예를 들어 설명할 수 있다.
- [ ] Okt로 형태소와 품사를 뽑고 품사 필터를 적용할 수 있다.
- [ ] 공백 vs 형태소 비교표를 만들고 **왜** 차이가 나는지 설명할 수 있다.
- [ ] 영어와 한국어의 토큰화 방식이 다른 이유를 언어 특성으로 설명할 수 있다.
- [ ] 분석 결과를 설정 지문과 함께 캐시할 수 있다.

## 🔗 과제 연결과 다음 장
- 9.7 비교표와 리포트용 정리는 과제 요구사항 "공백 토큰화를 쓰면 성능이 왜 달라지는지 분석"과 학습 목표 ①에 그대로 쓰입니다.
- **10장. 불용어와 전처리 파이프라인**에서 남은 기능어를 데이터 근거로 지우고, 정제·토큰화·불용어를 하나의 파이프라인으로 묶습니다.